# RBF Solver Benchmark — Dense vs Streaming OOC

Three approaches to solving the same RBF linear system `A x = b`:

| Method | Where matrix lives | Scales to? |
|--------|-------------------|------------|
| **scipy** | RAM (numpy FP64) | CPU RAM limit |
| **cupy** | VRAM (GPU FP64) | GPU VRAM limit |
| **MPDOK OOC** | Tiles streamed RAM→VRAM | Unlimited (SSD path) |

**§1** shows all three succeeding at small N.  
**§2** shows scipy/cupy hitting the memory wall while MPDOK OOC streaming keeps going.

In [1]:
# ═══════════════════════════════════════════════════════════════════════
#  USER CONFIGURATION — change N_SMALL / N_LARGE and re-run all cells
# ═══════════════════════════════════════════════════════════════════════

N_SMALL  = 8_192    # All methods fit here. Try: 4096, 8192, 12288, 16384
N_LARGE  = 32_768   # cupy OOMs here on an 8 GB GPU.
                    # Try: 24576, 32768, 49152
                    # Set N_LARGE=98304 + STORE='ssd' to see scipy also OOM.

STORE    = 'ram'    # 'ram'  — FP32 matrix cached in system RAM (fast)
                    # 'ssd'  — FP32 matrix streamed from NVMe (unlimited N)
SSD_PATH = '/var/home/fraser/mpdok_demo.bin'   # used only when STORE='ssd'

In [2]:
import sys, os, time, warnings, gc
import numpy as np
import cupy as cp
import psutil
from scipy.sparse.linalg import gmres as sp_gmres

# All modules are in the same folder as this notebook.
_here = os.path.abspath('')
if _here not in sys.path:
    sys.path.insert(0, _here)

from mpdok_ooc import MPDOKOOCSolver
from rbf_kernel import synthetic_coords, weather_front, _estimate_gamma

REG = 1e-2
TOL = 1e-5

dev       = cp.cuda.Device(0)
gpu_name  = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
vram_gb   = dev.mem_info[1] / 1024**3
ram_gb    = psutil.virtual_memory().total / 1024**3

def fp64_gb(N): return N**2 * 8 / 1024**3
def fp32_gb(N): return N**2 * 4 / 1024**3
def vram_free(): return dev.mem_info[0] / 1024**3
def ram_free():  return psutil.virtual_memory().available / 1024**3

print(f"GPU  : {gpu_name}  ({vram_gb:.1f} GB VRAM)")
print(f"RAM  : {ram_gb:.1f} GB total")
print()
print(f"N_SMALL = {N_SMALL:,}   FP64 = {fp64_gb(N_SMALL):.2f} GB   FP32 = {fp32_gb(N_SMALL):.2f} GB")
print(f"N_LARGE = {N_LARGE:,}   FP64 = {fp64_gb(N_LARGE):.2f} GB   FP32 = {fp32_gb(N_LARGE):.2f} GB")

GPU  : NVIDIA GeForce RTX 4060  (7.6 GB VRAM)
RAM  : 47.0 GB total

N_SMALL = 8,192   FP64 = 0.50 GB   FP32 = 0.25 GB
N_LARGE = 32,768   FP64 = 8.00 GB   FP32 = 4.00 GB


## §1 — All methods compete at N_SMALL

The full FP64 kernel matrix fits in both RAM and VRAM at this size.
All three solvers reach the same answer; the difference is *where* the matrix lives and *how fast* the matvec runs.

In [3]:
print(f"{'═'*62}")
print(f"  §1  N = {N_SMALL:,}   FP64 matrix = {fp64_gb(N_SMALL):.2f} GB")
print(f"{'═'*62}\n")

# Shared data
coords_s    = synthetic_coords(N_SMALL, seed=42)
b_s         = weather_front(coords_s, t=0)
coords_np_s = cp.asnumpy(coords_s)
b_np_s      = cp.asnumpy(b_s)
gamma_s     = float(_estimate_gamma(coords_s))

results_s = {}

# ── 1. scipy: full FP64 matrix in RAM, iterative GMRES ──────────────
print("1. scipy  (CPU GMRES, full FP64 matrix in RAM)")
t0 = time.perf_counter()
sq_np = np.sum(coords_np_s**2, axis=1)
D2_np = sq_np[:, None] + sq_np[None, :] - 2.0 * (coords_np_s @ coords_np_s.T)
np.maximum(D2_np, 0.0, out=D2_np)
A_np  = np.exp(-gamma_s * D2_np); del D2_np
np.fill_diagonal(A_np, A_np.diagonal() + REG)
t_build_sp = time.perf_counter() - t0

t0 = time.perf_counter()
with warnings.catch_warnings(record=True):
    warnings.simplefilter("always")
    x_sp, _ = sp_gmres(A_np, b_np_s, rtol=TOL, restart=50, maxiter=20*50)
t_solve_sp = time.perf_counter() - t0
rr_sp = float(np.linalg.norm(b_np_s - A_np @ x_sp) / np.linalg.norm(b_np_s))
del A_np; gc.collect()
ok_sp = rr_sp < TOL
print(f"   build {t_build_sp:.1f}s  solve {t_solve_sp:.1f}s  rel_res {rr_sp:.2e}  {'✓ OK' if ok_sp else '✗ NOT CONVERGED'}")
results_s['scipy GMRES'] = dict(build=t_build_sp, solve=t_solve_sp, rr=rr_sp, ok=ok_sp)

# ── 2. cupy: full FP64 matrix in VRAM, direct solve (cuSOLVER) ──────
print("\n2. cupy  (GPU cuSOLVER, full FP64 matrix in VRAM)")
t0 = time.perf_counter()
sq_c = cp.sum(coords_s**2, axis=1)
D2_c = sq_c[:, None] + sq_c[None, :] - 2.0 * (coords_s @ coords_s.T)
cp.maximum(D2_c, 0.0, out=D2_c)
A_cp = cp.exp(-gamma_s * D2_c); del D2_c
ki   = cp.arange(N_SMALL); A_cp[ki, ki] += REG
t_build_cp = time.perf_counter() - t0

t0 = time.perf_counter()
x_cp = cp.linalg.solve(A_cp, b_s)
cp.cuda.Device(0).synchronize()
t_solve_cp = time.perf_counter() - t0
rr_cp = float(cp.linalg.norm(b_s - A_cp @ x_cp) / cp.linalg.norm(b_s))
del A_cp; cp.get_default_memory_pool().free_all_blocks()
ok_cp = rr_cp < TOL
print(f"   build {t_build_cp:.1f}s  solve {t_solve_cp:.1f}s  rel_res {rr_cp:.2e}  {'✓ OK' if ok_cp else '✗ NOT CONVERGED'}")
results_s['cupy direct'] = dict(build=t_build_cp, solve=t_solve_cp, rr=rr_cp, ok=ok_cp)

# ── 3. MPDOK OOC: tiles streamed RAM→VRAM, GMRES-IR ─────────────────
tile_vram_mb = 4096 * N_SMALL * 4 / 1024**2
print(f"\n3. MPDOK OOC streaming  (max {tile_vram_mb:.0f} MB VRAM at any moment)")
ooc_s = MPDOKOOCSolver(tile_rows=4096)
t0 = time.perf_counter()
ooc_s.build(coords_s, gamma=gamma_s, reg=REG, store='ram', verbose=False)
t_build_ooc = time.perf_counter() - t0

t0 = time.perf_counter()
with warnings.catch_warnings(record=True) as w_ooc:
    warnings.simplefilter("always")
    x_ooc_s = ooc_s.solve(b_s, tol=TOL, maxiter_outer=20, restart=50, verbose=False)
    conv_ooc = len(w_ooc) == 0
t_solve_ooc = time.perf_counter() - t0
rr_ooc = float(cp.linalg.norm(b_s - ooc_s._tiled_dgemv_fp64(x_ooc_s)) / cp.linalg.norm(b_s))
ooc_s.free()
ok_ooc = conv_ooc and rr_ooc < TOL
print(f"   build {t_build_ooc:.1f}s  solve {t_solve_ooc:.1f}s  rel_res {rr_ooc:.2e}  {'✓ OK' if ok_ooc else '✗ NOT CONVERGED'}")
results_s['MPDOK OOC'] = dict(build=t_build_ooc, solve=t_solve_ooc, rr=rr_ooc, ok=ok_ooc)

# ── Summary table ────────────────────────────────────────────────────
print(f"\n{'═'*62}")
print(f"  {'Method':<20} {'Build':>7} {'Solve':>7} {'Total':>7}  {'rel_res':>9}")
print(f"  {'─'*60}")
for name, r in results_s.items():
    tot = r['build'] + r['solve']
    mark = '✓' if r['ok'] else '✗'
    print(f"  {name:<20} {r['build']:>6.1f}s {r['solve']:>6.1f}s {tot:>6.1f}s  {r['rr']:>9.2e}  {mark}")
print(f"{'═'*62}")

══════════════════════════════════════════════════════════════
  §1  N = 8,192   FP64 matrix = 0.50 GB
══════════════════════════════════════════════════════════════

1. scipy  (CPU GMRES, full FP64 matrix in RAM)
   build 4.4s  solve 0.6s  rel_res 6.16e-06  ✓ OK

2. cupy  (GPU cuSOLVER, full FP64 matrix in VRAM)
   build 0.0s  solve 1.8s  rel_res 7.88e-14  ✓ OK

3. MPDOK OOC streaming  (max 128 MB VRAM at any moment)
   build 0.1s  solve 3.0s  rel_res 7.16e-07  ✓ OK

══════════════════════════════════════════════════════════════
  Method                 Build   Solve   Total    rel_res
  ────────────────────────────────────────────────────────────
  scipy GMRES             4.4s    0.6s    5.0s   6.16e-06  ✓
  cupy direct             0.0s    1.8s    1.8s   7.88e-14  ✓
  MPDOK OOC               0.1s    3.0s    3.1s   7.16e-07  ✓
══════════════════════════════════════════════════════════════


## §2 — The memory wall at N_LARGE

Dense methods require **N² memory all at once**.  
MPDOK OOC streaming requires only **one tile (tile_rows × N)** in VRAM at any moment;
the full FP32 matrix lives in RAM (or on NVMe when `STORE='ssd'`).

| Method | Memory needed | Limit on this machine |
|--------|--------------|----------------------|
| scipy | N² × 8 bytes RAM | ~69K before OOM |
| cupy | N² × 8 bytes VRAM | ~28K before OOM |
| MPDOK OOC (ram) | tile × 4 bytes VRAM + N² × 4 bytes RAM | ~108K before OOM |
| MPDOK OOC (ssd) | tile × 4 bytes VRAM + NVMe space | unlimited |

Set `N_LARGE = 98_304` and `STORE = 'ssd'` to see scipy also run out of RAM.

In [4]:
fp64_l    = fp64_gb(N_LARGE)
fp32_l    = fp32_gb(N_LARGE)
vf        = vram_free()
rf        = ram_free()
tile_mb_l = 4096 * N_LARGE * 4 / 1024**2

print(f"{'═'*65}")
print(f"  §2  N = {N_LARGE:,}   FP64 = {fp64_l:.1f} GB   FP32 = {fp32_l:.1f} GB")
print(f"      VRAM free: {vf:.1f} GB   RAM free: {rf:.1f} GB")
print(f"      MPDOK OOC tile: {tile_mb_l:.0f} MB  (one row-band in VRAM at a time)")
print(f"{'═'*65}\n")

coords_l    = synthetic_coords(N_LARGE, seed=42)
b_l         = weather_front(coords_l, t=0)
b_np_l      = cp.asnumpy(b_l)
coords_np_l = cp.asnumpy(coords_l)
gamma_l     = float(_estimate_gamma(coords_l))

# ── 1. scipy ─────────────────────────────────────────────────────────
print(f"1. scipy  (needs {fp64_l:.1f} GB RAM)")
sp_time = None
if fp64_l > rf * 0.85:
    print(f"   ✗ OUT OF MEMORY  ({fp64_l:.1f} GB > {rf:.1f} GB available RAM)")
else:
    try:
        print(f"   Allocating {fp64_l:.1f} GB in RAM ...", end=' ', flush=True)
        t0 = time.perf_counter()
        sq_np_l = np.sum(coords_np_l**2, axis=1)
        D2_np_l = sq_np_l[:, None] + sq_np_l[None, :] - 2.0 * (coords_np_l @ coords_np_l.T)
        np.maximum(D2_np_l, 0.0, out=D2_np_l)
        A_np_l  = np.exp(-gamma_l * D2_np_l); del D2_np_l
        np.fill_diagonal(A_np_l, A_np_l.diagonal() + REG)
        t_build_sp_l = time.perf_counter() - t0
        print(f"done in {t_build_sp_l:.1f}s")
        t0 = time.perf_counter()
        with warnings.catch_warnings(record=True):
            warnings.simplefilter("always")
            x_sp_l, _ = sp_gmres(A_np_l, b_np_l, rtol=TOL, restart=50, maxiter=20*50)
        t_solve_sp_l = time.perf_counter() - t0
        rr_sp_l = float(np.linalg.norm(b_np_l - A_np_l @ x_sp_l) / np.linalg.norm(b_np_l))
        del A_np_l; gc.collect()
        sp_time = t_build_sp_l + t_solve_sp_l
        print(f"   build {t_build_sp_l:.1f}s  solve {t_solve_sp_l:.1f}s  rel_res {rr_sp_l:.2e}")
        scipy_oom_n = int((rf * 0.85 * 1024**3 / 8) ** 0.5)
        print(f"   (matrix fits here — OOM above N ≈ {scipy_oom_n:,} on this machine)")
    except MemoryError:
        print("   ✗ OUT OF MEMORY")

# ── 2. cupy ──────────────────────────────────────────────────────────
print(f"\n2. cupy  (needs {fp64_l:.1f} GB VRAM, {vf:.1f} GB available)")
if fp64_l > vf * 0.85:
    print(f"   ✗ OUT OF MEMORY  ({fp64_l:.1f} GB > {vf:.1f} GB VRAM)")
else:
    try:
        print(f"   Allocating {fp64_l:.1f} GB in VRAM ...", end=' ', flush=True)
        sq_c_l = cp.sum(coords_l**2, axis=1)
        D2_c_l = sq_c_l[:, None] + sq_c_l[None, :] - 2.0 * (coords_l @ coords_l.T)
        cp.maximum(D2_c_l, 0.0, out=D2_c_l)
        A_cp_l  = cp.exp(-gamma_l * D2_c_l); del D2_c_l
        ki_l    = cp.arange(N_LARGE); A_cp_l[ki_l, ki_l] += REG
        print("done")
        t0 = time.perf_counter()
        x_cp_l = cp.linalg.solve(A_cp_l, b_l)
        cp.cuda.Device(0).synchronize()
        t_solve_cp_l = time.perf_counter() - t0
        rr_cp_l = float(cp.linalg.norm(b_l - A_cp_l @ x_cp_l) / cp.linalg.norm(b_l))
        del A_cp_l; cp.get_default_memory_pool().free_all_blocks()
        print(f"   solve {t_solve_cp_l:.1f}s  rel_res {rr_cp_l:.2e}")
    except Exception:
        cp.get_default_memory_pool().free_all_blocks()
        print(f"   ✗ OUT OF MEMORY")

# ── 3. MPDOK OOC ─────────────────────────────────────────────────────
maxiter_l = max(20, 6 + int((N_LARGE / 8192 - 1) * 3))
print(f"\n3. MPDOK OOC streaming  ({tile_mb_l:.0f} MB VRAM, FP32 in {STORE.upper()})")
ooc_l = MPDOKOOCSolver(tile_rows=4096)

t0 = time.perf_counter()
ooc_l.build(coords_l, gamma=gamma_l, reg=REG, store=STORE,
            path=SSD_PATH if STORE == 'ssd' else None, verbose=True)
t_build_l = time.perf_counter() - t0
print(f"   build: {t_build_l:.1f}s")

t0 = time.perf_counter()
with warnings.catch_warnings(record=True) as w_l:
    warnings.simplefilter("always")
    x_ooc_l = ooc_l.solve(b_l, tol=TOL, maxiter_outer=maxiter_l, restart=50, verbose=True)
    conv_l = len(w_l) == 0
t_solve_l = time.perf_counter() - t0

rr_l = float(cp.linalg.norm(b_l - ooc_l._tiled_dgemv_fp64(x_ooc_l)) / cp.linalg.norm(b_l))
ok_l = conv_l and rr_l < TOL
ooc_l.free()
if STORE == 'ssd' and os.path.exists(SSD_PATH):
    os.remove(SSD_PATH)
    print(f"   Removed {SSD_PATH}")

# ── Final summary ─────────────────────────────────────────────────────
print(f"\n{'═'*65}")
print(f"  §2 results at N = {N_LARGE:,}")
print(f"  {'─'*63}")
sp_str = f"✓ {sp_time:.0f}s" if sp_time else "✗ OOM"
print(f"  scipy GMRES   {sp_str:>10}   (matrix: {fp64_l:.1f} GB RAM)")
cp_oom = fp64_l > vf * 0.85
print(f"  cupy direct   {'✗ OOM':>10}   (matrix: {fp64_l:.1f} GB VRAM, {vf:.1f} GB free)" if cp_oom
      else f"  cupy direct   {'✓':>10}")
ooc_str = f"✓ {t_build_l + t_solve_l:.0f}s" if ok_l else "✗ NOT CONV"
print(f"  MPDOK OOC     {ooc_str:>10}   (tile: {tile_mb_l:.0f} MB VRAM, rel_res {rr_l:.2e})")
print(f"{'═'*65}")
print(f"\n  Tip: set N_LARGE = 98_304 and STORE = 'ssd' to see scipy also OOM ({98304**2*8/1024**3:.0f} GB RAM)")
print(f"       while MPDOK OOC streams {98304**2*4/1024**3:.0f} GB FP32 from NVMe.")

═════════════════════════════════════════════════════════════════
  §2  N = 32,768   FP64 = 8.0 GB   FP32 = 4.0 GB
      VRAM free: 6.3 GB   RAM free: 38.6 GB
      MPDOK OOC tile: 512 MB  (one row-band in VRAM at a time)
═════════════════════════════════════════════════════════════════

1. scipy  (needs 8.0 GB RAM)
   Allocating 8.0 GB in RAM ... done in 73.6s
   build 73.6s  solve 9.8s  rel_res 7.44e-06
   (matrix fits here — OOM above N ≈ 66,382 on this machine)

2. cupy  (needs 8.0 GB VRAM, 6.3 GB available)
   ✗ OUT OF MEMORY  (8.0 GB > 6.3 GB VRAM)

3. MPDOK OOC streaming  (512 MB VRAM, FP32 in RAM)
  OOC build: N=32,768  gamma=1.505e-04  FP32=4.00 GB  tiles=8  store=ram
    tile 8/8  (100%)
  build done in 2.0 s
   build: 2.0s
  outer 0: rel_res=1.00e+00  (FP64 DGEMV 0.6s)
           inner GMRES 19.8s
  outer 1: rel_res=5.27e-05  (FP64 DGEMV 0.6s)
           inner GMRES 21.6s
  outer 2: rel_res=2.60e-06  (FP64 DGEMV 0.6s)

════════════════════════════════════════════════════════